<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/09-serving-inference/01-serving-frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Serving Frameworks (concept)

**Goal:** Understand the inference-serving stack an AI engineer actually picks between — vLLM, TGI, Triton, TensorRT-LLM — what each optimizes, how they map onto the raw API you've been calling all along, and when to reach for which.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Why this is concept-first (and what runs)

Every notebook so far called a model over an API — Groq hosts the model, you send JSON. This section is about **the thing on the other side of that API**: the server that loads the weights onto a GPU and turns your request into tokens. That's the layer an AI engineer working on *AI distributed systems* owns.

Groq can't teach it hands-on — it *is* a managed serving endpoint, so there's nothing to stand up. Self-hosting vLLM/Triton/TensorRT-LLM needs a GPU. So this notebook is **concept-first**: the body is the mental model and the decision — no GPU required — and an **optional, clearly fenced appendix** at the end stands up a real vLLM server on a free Colab T4 for those who want to see it. Same pattern as the [section-06 LoRA appendix](../06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb).

> **⭐ Key takeaway —** you've been the *client* of an inference server this whole time. Serving frameworks are the *server*. Knowing what they optimize is what lets you reason about throughput, latency, and cost — the questions an AI-systems interview actually asks.

## The one API, two very different servers behind it

The request you send is identical whether Groq, OpenAI, or your own box answers it — an OpenAI-compatible `chat.completions` call. What differs is what happens **after** the request lands:

```
your request ─▶ [ serving framework ] ─▶ GPU: load weights, run forward passes ─▶ tokens ─▶ you
                        ▲
              this is what section 09 is about
```

A serving framework's whole job is to answer *many* of those requests on *expensive* hardware **efficiently**. That single constraint — a GPU costs real money by the second, and one request barely uses it — drives every optimization in the next notebook (batching, KV cache, quantization). Here we map the landscape first.

## The four you should be able to name

These are the names that come up in AI-systems interviews and job posts. You don't need to have run all four — you need to know *what each is for* and *how they relate*.

| Framework | Who makes it | Optimizes for | The one-liner |
|---|---|---|---|
| **vLLM** | UC Berkeley → community | **Throughput** on general GPUs | The default open-source choice. PagedAttention manages the KV cache like OS virtual memory, so it serves far more concurrent requests per GPU. OpenAI-compatible server out of the box. |
| **TGI** (Text Generation Inference) | Hugging Face | Easy, production-ready serving | Batteries-included: metrics, safetensors, tight HF-ecosystem fit. The "just serve my HF model well" option. |
| **TensorRT-LLM** | NVIDIA | **Lowest latency** on NVIDIA GPUs | Compiles the model into a hardware-specific engine — fastest per-token, but the build step is heavy and NVIDIA-only. |
| **Triton Inference Server** | NVIDIA | Serving *anything* at scale | Not LLM-specific — a general model server (LLMs, vision, classic ML) with a TensorRT-LLM backend. The orchestration layer, not the engine. |

> **💡 Why so many —** they sit at different layers, not in a straight ranking. Triton is a *server* that can run a TensorRT-LLM *engine* as one of its backends; vLLM and TGI are self-contained servers with their own engines. "Best" only means anything once you name the metric — throughput, latency, or operational simplicity.

## How it maps to the loop you already built

Nothing here replaces what you know — it sits *under* it. Recall the agent loop (section 05): you sent `messages`, got back a response, sent the next request. Every one of those calls hit a serving framework. Swapping Groq for a self-hosted vLLM server changes exactly one thing in your code:

```python
# Section 01–05: Groq's managed server
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=...)

# Same code, self-hosted vLLM (started with: python -m vllm.entrypoints.openai.api_server --model ...)
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# ...everything downstream — tool calling, the agent loop, structured output — is byte-for-byte identical.
```

That's the payoff of the repo's OpenAI-compatible spine: **the serving layer is swappable**. vLLM, TGI, and Triton all expose the same OpenAI-compatible endpoint, so the framework-free patterns you built transfer to a self-hosted deployment without a rewrite.

## When an AI engineer reaches for which

The decision, the way it actually gets made on a team:

- **Prototyping / "just serve this HF model"** → **TGI** or **vLLM**. Least setup, OpenAI-compatible server in one command.
- **Maximize requests-per-GPU-dollar (most self-hosted LLM serving)** → **vLLM**. Its continuous batching + PagedAttention is why it's the community default; this is the common answer.
- **Squeeze out the last millisecond of latency, NVIDIA hardware, willing to pay build complexity** → **TensorRT-LLM** (often *inside* Triton).
- **One serving platform for many model types (LLM + vision + classic ML), multi-model, versioned, at org scale** → **Triton**.
- **You just need an API and don't want to run GPUs at all** → a managed endpoint (Groq, OpenAI, Bedrock, Together, Fireworks). Most products start here and self-host only when cost or control demands it.

> **⚠️ Production reality —** self-hosting is a cost/control decision, not a default. You take on GPUs, autoscaling, and on-call in exchange for lower per-token cost at scale, data residency, or custom models. Below some volume, a managed endpoint is cheaper *and* less work. "We used a managed API until volume justified self-hosting on vLLM" is a stronger interview answer than "we ran our own GPUs from day one."

For the systems view — how these servers sit behind a load balancer, autoscale, and get sized — see the next section, [ML system design](../10-ml-system-design/01-designing-an-inference-service.ipynb).

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Keep an OpenAI-compatible client seam so the serving layer stays swappable | Hard-code one vendor's SDK so a serving change is a rewrite |
| Name the metric first (throughput / latency / ops simplicity), then pick | Ask "which framework is best?" with no metric in mind |
| Default to a managed endpoint; self-host when cost/control/data justify it | Stand up your own GPUs on day one because it "looks more serious" |
| Reach for vLLM for throughput-per-dollar; TensorRT-LLM for lowest latency | Assume TensorRT-LLM is "just faster" and ignore its build cost + NVIDIA lock-in |
| Know Triton is a *server* that can host a TensorRT-LLM *engine* (different layers) | Compare Triton vs TensorRT-LLM as if they were competing engines |

## Optional appendix — stand up a real vLLM server (needs a GPU)

> **⚠️ This appendix needs a GPU runtime and is skippable.** In Colab: **Runtime → Change runtime type → T4 GPU**. It does **not** use Groq — it downloads a small model and serves it locally. On the free CPU runtime, read it and move on; the concepts above are the graded material.

The cells install vLLM, launch its OpenAI-compatible server on a small model, and hit it with the *exact* client code from section 01 — proving the serving layer is swappable. **Self-hosting is a real dependency-and-resource exercise, not a one-click install** — that's the section-09 lesson, so the cells below are written to *surface* problems instead of hiding them. Two gotchas you may hit, both handled here:

- **A CUDA version clash on import** — installing vLLM upgrades PyTorch, which can leave Colab's pre-installed `torchaudio` compiled for a different CUDA version. `transformers` imports `torchaudio` unconditionally, so the mismatch crashes the server *at import*. Fix: uninstall `torchaudio` (text inference never uses it).
- **"Free memory … is less than desired GPU memory utilization"** — a leftover vLLM process from a previous attempt still holds the GPU. Only one vLLM can own the card at a time. Fix: kill stale processes (or **Runtime → Restart session**) and cap `--gpu-memory-utilization`.

In [ ]:
# Appendix cell 1 — GPU check. Stop here (don't run the rest) if this prints "no GPU".
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "no GPU — switch runtime to T4, or skip this appendix (concepts above are what's graded)")

In [ ]:
# Appendix cell 2 — install vLLM (a few minutes on first run; GPU only), then fix the torchaudio
# CUDA-version clash. vLLM upgrades PyTorch; Colab's stale torchaudio then fails to import and would
# crash the server. Text inference doesn't use torchaudio, so we remove it. Harmless if not present.
%pip install -q vllm
!pip uninstall -y torchaudio 2>/dev/null || true
# Verify the imports the server needs actually load (this is what silently failed before):
!python -c "import transformers, vllm; print('imports OK — vllm', vllm.__version__)"

# On a fresh runtime this import check passes with no restart needed. Rarely, if pip upgraded a
# package already loaded in this kernel and the line above errors, do Runtime -> Restart session,
# then rerun from the GPU-check cell, SKIPPING this %pip line (the packages are already installed).

In [ ]:
# Appendix cell 3 — launch the OpenAI-compatible server in the BACKGROUND, with logs visible.
# Key choices, each learned the hard way:
#   --enforce-eager             : skip CUDA-graph compile (faster startup, avoids a class of T4 crashes)
#   --gpu-memory-utilization .85: don't grab 92%; leave headroom so startup doesn't fail on a busy card
#   logs -> a file we can tail  : if the engine dies, we PRINT why instead of timing out blindly
import subprocess, time, urllib.request

LOG = "/content/vllm.log"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
     "--max-model-len", "2048", "--port", "8000",
     "--enforce-eager", "--gpu-memory-utilization", "0.85"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

up = False
for _ in range(90):                       # up to ~7.5 min (model download + load)
    if server.poll() is not None:         # process died — show the real cause
        print(f"server exited (code {server.returncode}). Root cause in the log tail:\n")
        print("".join(open(LOG).readlines()[-25:]))
        break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("vLLM server is up on :8000"); up = True; break
    except Exception:
        time.sleep(5)
if not up and server.poll() is None:
    print("still starting (downloading / loading). Log tail:\n")
    print("".join(open(LOG).readlines()[-15:]))

# Common failure: "Free memory ... less than desired GPU memory utilization" means a stale vLLM
# still holds the card. Run  !nvidia-smi  to see it, then  Runtime -> Restart session  and rerun.

In [ ]:
# Appendix cell 4 — the punchline: the SAME client code from section 01, pointed at YOUR server.
#
# What to expect: cell 3 prints "up" as soon as the HTTP layer answers /v1/models, but the FIRST
# real request can still race the engine's final warm-up — you may see a connection error for a few
# seconds. So we RETRY with backoff instead of failing on the first miss. If the server process has
# actually died (e.g. the engine crashed after startup), we detect that and print the log tail.
import time
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")  # local vLLM, no key needed

resp = None
for attempt in range(1, 13):                      # ~1 minute of retries
    if 'server' in globals() and server.poll() is not None:   # process gone -> not a race, a crash
        print(f"server process exited (code {server.returncode}). Log tail:\n")
        print("".join(open("/content/vllm.log").readlines()[-25:]))
        break
    try:
        resp = client.chat.completions.create(
            model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            messages=[{"role": "user", "content": "In one sentence, what does a serving framework do?"}],
            max_tokens=64)
        break                                     # success
    except Exception as e:
        print(f"attempt {attempt}: not ready yet ({type(e).__name__}) — retrying in 5s")
        time.sleep(5)

if resp:
    print("\n" + resp.choices[0].message.content)
    print("\nusage:", resp.usage)                 # same response shape you've parsed since section 00
    # (TinyLlama is 1.1B — expect a rough answer. The point is the SEAM, not the model quality.)
else:
    print("\nno response — rerun cell 3 (it cleans up and relaunches), then rerun this cell.")

In [ ]:
# Appendix cell 5 — ALWAYS free the GPU when done. Only one vLLM can own the card; a leftover
# process is exactly what causes the "free memory" error on your next run.
server.terminate()
import time; time.sleep(3)
!pkill -9 -f vllm.entrypoints 2>/dev/null || true   # belt-and-suspenders: kill any stray workers
!nvidia-smi --query-gpu=memory.used --format=csv,noheader   # should drop back toward ~0 MiB

## Exercises

1. **Map your own stack.** Take the capstone shape you're planning (section 12). Write down: managed endpoint or self-hosted? If self-hosted, which of the four frameworks and *why* — name the metric that decides it. Two sentences; this is the exact answer an interviewer wants.
2. **The swap in code.** Without running anything, write the two-line diff that moves the section-05 agent loop from Groq to a self-hosted vLLM server. Confirm nothing else in the loop changes — and say in one sentence why that's true.
3. **Layers, not a ranking.** In your own words, explain why "Triton vs TensorRT-LLM" is a category error, but "vLLM vs TGI" is a fair comparison. (Hint: which one is a *server* and which is an *engine*?)
4. **(If you ran the appendix.)** Change the appendix model to a second small model, restart the server, and hit it with the same client. Note what *didn't* change in your code — that invariance is the whole point of the OpenAI-compatible seam.